# Create Canada Council Prizes Awards (PRIZE PATTERN)

Canada Council for the Arts prize laureates from official Canada Council sources.

**Prerequisite:** run `scripts/local/canada_council_prizes_to_s3.py` first. Contractor local validation used `--skip-upload`; admin must run the script with S3 credentials and then run this notebook in Databricks.

**Source:** `https://canadacouncil.ca/funding/prizes`, official Canada Council cumulative-winner PDFs linked from those prize pages, and the official GGBooks JSON backing `https://ggbooks.ca/past-winners-and-finalists` for Governor General's Literary Awards.

**S3 location:** `s3a://openalex-ingest/awards/canada_council_prizes/canada_council_prizes.parquet`

**Awarding body:** Canada Council for the Arts - OpenAlex `F4320319951`

Funder details from Step 0:

- funder_id: `4320319951`
- display_name: `Canada Council for the Arts`
- ror_id: `https://ror.org/02kzsfv42`
- doi: `10.13039/501100000195`

## Prize-pattern modeling

- One row per `(prize x year x laureate)` when the official source identifies distinct laureates. GGBooks co-author/co-illustrator strings are split into one row per named laureate.
- `lead_investigator` is the laureate. Organization-like recipients are preserved as the source publishes them, with `family_name` left NULL.
- `funding_type = 'prize'`.
- `provenance = 'canada_council_prizes'`.
- `funder_scheme` is the prize name plus category/language when the source exposes them.
- Amounts are set to NULL and `currency = CAD`. The prize pages expose current public amounts, but many cumulative lists span decades and do not publish historical per-laureate amount or shared apportionment. The raw parquet preserves `source_amount_text` for audit, so this is an explicit prize-pattern Step 6.7 waiver.
- Local dry-run on 2026-05-19 emitted 2,180 rows: 836 GGBooks JSON rows and 1,344 high-confidence PDF rows, with 12 official prize pages skipped into the local skipped-source manifest because their PDFs require dedicated parsers or have no cumulative source.

## Admin handoff

No `jobs/create_funder_sourced_awards.yaml` task is added in this PR. Admin must upload the parquet, run this notebook, inspect QA output, run `CreateAwards.ipynb`, and only then wire the recurring job if approved.


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.canada_council_prizes_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/canada_council_prizes/canada_council_prizes.parquet`;


## Step 1.5: Inspect raw data before transform

Runbook requirement: verify actual source columns and inspect money/currency-shaped values before writing or running transformation SQL. `source_amount_text` is intentionally audit-only for this prize source; transformed `amount` is NULL with the documented prize waiver above.


In [ ]:
%sql
SELECT COUNT(*) AS raw_rows
FROM openalex.awards.canada_council_prizes_raw;


In [ ]:
%sql
DESCRIBE openalex.awards.canada_council_prizes_raw;


In [ ]:
%sql
SELECT *
FROM openalex.awards.canada_council_prizes_raw
LIMIT 10;


In [ ]:
%sql
SELECT
    source_type,
    parser_name,
    parse_confidence,
    COUNT(*) AS rows,
    MIN(TRY_CAST(year AS INT)) AS min_year,
    MAX(TRY_CAST(year AS INT)) AS max_year
FROM openalex.awards.canada_council_prizes_raw
GROUP BY source_type, parser_name, parse_confidence
ORDER BY source_type, parser_name;


In [ ]:
%sql
-- Money/currency field discovery per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'canada_council_prizes_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp|currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT
    source_amount_text,
    currency,
    COUNT(*) AS rows
FROM openalex.awards.canada_council_prizes_raw
GROUP BY source_amount_text, currency
ORDER BY rows DESC, source_amount_text
LIMIT 50;


## Step 1.6: Funder existence fail-fast

In [ ]:
%sql
-- Must return exactly one row. If this returns 0 rows, STOP.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320319951;


## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.canada_council_prizes_awards
USING delta
AS
WITH canada_council_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319951
), normalized AS (
    SELECT
        *,
        TRY_CAST(year AS INT) AS award_year_int,
        NULLIF(TRIM(funder_award_id), '') AS funder_award_id_norm,
        NULLIF(TRIM(prize_name), '') AS prize_name_norm,
        NULLIF(TRIM(category), '') AS category_norm,
        NULLIF(TRIM(language), '') AS language_norm,
        NULLIF(TRIM(laureate_name), '') AS laureate_name_norm,
        NULLIF(TRIM(laureate_given_name), '') AS given_name_norm,
        NULLIF(TRIM(laureate_family_name), '') AS family_name_norm,
        LOWER(NULLIF(TRIM(is_organization_like), '')) = 'true' AS is_organization_like_bool,
        NULLIF(TRIM(work_title), '') AS work_title_norm,
        NULLIF(TRIM(source_description), '') AS source_description_norm,
        NULLIF(TRIM(raw_entry_text), '') AS raw_entry_text_norm,
        NULLIF(TRIM(prize_url), '') AS prize_url_norm,
        NULLIF(TRIM(currency), '') AS currency_norm
    FROM openalex.awards.canada_council_prizes_raw
), awards AS (
    SELECT
        abs(xxhash64(CONCAT(
            TRY_CAST(f.funder_id AS STRING), ':canada-council:', LOWER(n.funder_award_id_norm)
        ))) % 9000000000 AS id,
        CONCAT(
            CAST(n.award_year_int AS STRING), ' ', n.prize_name_norm, ' - ', n.laureate_name_norm,
            CASE WHEN n.work_title_norm IS NOT NULL THEN CONCAT(' (', n.work_title_norm, ')') ELSE '' END
        ) AS display_name,
        NULLIF(CONCAT_WS(
            ' ',
            n.source_description_norm,
            CASE WHEN n.category_norm IS NOT NULL THEN CONCAT('Category: ', n.category_norm, '.') END,
            CASE WHEN n.language_norm IS NOT NULL THEN CONCAT('Language: ', n.language_norm, '.') END,
            CASE WHEN n.work_title_norm IS NOT NULL THEN CONCAT('Winning work: ', n.work_title_norm, '.') END,
            CASE WHEN n.raw_entry_text_norm IS NOT NULL THEN CONCAT('Source entry: ', n.raw_entry_text_norm) END
        ), '') AS description,
        f.funder_id,
        n.funder_award_id_norm AS funder_award_id,
        CAST(NULL AS DOUBLE) AS amount,
        COALESCE(n.currency_norm, 'CAD') AS currency,
        struct(
            CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
            f.display_name, f.ror_id, f.doi
        ) AS funder,
        'prize' AS funding_type,
        CONCAT_WS(' - ', n.prize_name_norm, n.category_norm, n.language_norm) AS funder_scheme,
        'canada_council_prizes' AS provenance,
        TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-01-01'), 'yyyy-MM-dd') AS start_date,
        TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-12-31'), 'yyyy-MM-dd') AS end_date,
        n.award_year_int AS start_year,
        n.award_year_int AS end_year,
        struct(
            CASE
                WHEN n.is_organization_like_bool THEN n.laureate_name_norm
                ELSE COALESCE(n.given_name_norm, n.laureate_name_norm)
            END AS given_name,
            CASE
                WHEN n.is_organization_like_bool THEN CAST(NULL AS STRING)
                ELSE n.family_name_norm
            END AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,
        n.prize_url_norm AS landing_page_url,
        CAST(NULL AS STRING) AS doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G',
            TRY_CAST(abs(xxhash64(CONCAT(
                TRY_CAST(f.funder_id AS STRING), ':canada-council:', LOWER(n.funder_award_id_norm)
            ))) % 9000000000 AS STRING)
        ) AS works_api_url,
        current_timestamp() AS created_date,
        current_timestamp() AS updated_date
    FROM normalized n
    CROSS JOIN canada_council_funder f
    WHERE n.funder_award_id_norm IS NOT NULL
      AND n.award_year_int IS NOT NULL
      AND n.prize_name_norm IS NOT NULL
      AND n.laureate_name_norm IS NOT NULL
)
SELECT *
FROM awards;


## Step 3: Delete old data and insert into `openalex_awards_raw`

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'canada_council_prizes' AND priority = 74;

-- Insert into openalex_awards_raw with priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    74 AS priority  -- Canada Council Prizes priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.canada_council_prizes_awards;


## Verification queries

In [ ]:
%sql
SELECT COUNT(*) AS total
FROM openalex.awards.canada_council_prizes_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.canada_council_prizes_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_display_name,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_lead_investigator,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_display_name,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_start_date
FROM openalex.awards.canada_council_prizes_awards;


In [ ]:
%sql
-- Prize-pattern amount waiver: amount is NULL by design, currency should be CAD.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS has_currency,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.canada_council_prizes_awards;


In [ ]:
%sql
SELECT funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.canada_council_prizes_awards
GROUP BY funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.canada_council_prizes_awards
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 30;


In [ ]:
%sql
SELECT id, COUNT(*) AS rows
FROM openalex.awards.canada_council_prizes_awards
GROUP BY id
HAVING COUNT(*) > 1;


In [ ]:
%sql
SELECT *
FROM openalex.awards.canada_council_prizes_awards
ORDER BY start_year DESC, funder_scheme, display_name
LIMIT 25;
